# Decision Layer — Decide → Act (C2 / C3)

Companion to notebook 08 (the Record layer). This shows the back half of the Record → Decide → Act pipeline (Exhibit 14 §9) end to end, in-process:

- **Decide (C2)** — a policy engine reads the recorded verdict trail and raises durable **alerts** via configurable rules (identity burst, sustained ICS anomalies, single high-severity events), weighted by each subject's *historical* outcomes.
- **Act (C3)** — when an alert fires, severity-routed **responders** run (log / ticket / webhook) and every response is recorded for audit; analysts **close** alerts.

Self-contained and deterministic (fresh throwaway DB; real backend via `TestClient`).

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'backend'))
DEMO_DB = ROOT / 'data' / 'verdicts_demo.db'
if DEMO_DB.exists(): DEMO_DB.unlink()
os.environ['VERDICT_DB'] = str(DEMO_DB)
from fastapi.testclient import TestClient
import app as appmod
import pandas as pd
client = TestClient(appmod.app)
print('backend ready:', client.get('/health').json())

C:\Users\User\ai-cybersecurity-portfolio\venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
C:\Users\User\ai-cybersecurity-portfolio\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4671.07it/s]

backend ready: {'status': 'ok'}


## 1. Generate activity the rules should catch
An attacker account authenticating across many machines (a behavioural burst = lateral movement), and a run of spiked ICS sensor readings.

In [2]:
ATTACKER = 'ANONYMOUS LOGON@C9999'
for i in range(12):
    client.post('/identity/score', json={'src_user':ATTACKER,'src_pc':f'C80{i:02d}',
        'auth_type':'TotallyUnrecognizedAuth','logon_type':'RemoteInteractive',
        'orientation':'LogOn','success':'Fail'})
ex = client.get('/ics/example').json()
for i in range(4):
    s = dict(ex); s['P1_LIT01'] = 5000.0 + i
    client.post('/ics/score', json={'readings': s})
print('streamed: 12 burst logins + 4 ICS attacks')

streamed: 12 burst logins + 4 ICS attacks


## 2. Decide — run the policy rules; see the alerts raised

In [3]:
created = client.post('/decision/evaluate').json()
print('evaluate created alert ids:', created['created_alert_ids'])
alerts = client.get('/decision/alerts?auto_evaluate=false&limit=50').json()
df = pd.DataFrame(alerts)[['id','rule','model','subject','severity','verdict_count','status']]
df

[ACTION:log] alert 1 :: [HIGH] identity_burst (identity/ANONYMOUS LOGON@C9999): 12 flagged logins from ANONYMOUS LOGON@C9999 within 300s contributing verdict(s)


[ACTION:log] alert 2 :: [HIGH] ics_sustained (ics/None): 4 sustained ICS anomalies within 300s contributing verdict(s)


[ACTION:log] alert 3 :: [HIGH] high_severity (ics/ics): single verdict with an extreme anomaly score contributing verdict(s)


evaluate created alert ids: [1, 2, 3]


,id,rule,model,subject,severity,verdict_count,status
0,3,high_severity,ics,ics,high,1,open
1,2,ics_sustained,ics,NaN,high,4,open
2,1,identity_burst,identity,ANONYMOUS LOGON@C9999,high,12,open


## 3. Act — the responses each alert triggered (audited)
High-severity alerts route to `log` + `ticket` (+ `webhook` when `ACTION_WEBHOOK_URL` is configured; skipped here). Every response is recorded.

In [4]:
acts = client.get('/decision/actions?limit=50').json()
adf = pd.DataFrame(acts)[['alert_id','action_type','status']]
print(adf.to_string(index=False))
# a ticket stub, with its generated reference and title
tkt = next(a for a in acts if a['action_type']=='ticket')
print('\nexample ticket:', tkt['detail'])

 alert_id action_type  status
        3     webhook skipped
        3      ticket      ok
        3         log      ok
        2     webhook skipped
        2      ticket      ok
        2         log      ok
        1     webhook skipped
        1      ticket      ok
        1         log      ok

example ticket: {'ticket_ref': 'SEC-3', 'title': '[HIGH] high_severity (ics/ics): single verdict with an extreme anomaly score contributing verdict(s)', 'system': 'stub'}


## 4. Alert lifecycle — analyst closes an alert
Closing removes it from the open queue and is *sticky*: the rule won't re-fire for that subject within the window.

In [5]:
aid = alerts[0]['id']
print('close:', client.post(f'/decision/alerts/{aid}/close').json())
open_after = [a['id'] for a in client.get('/decision/alerts?status=open&auto_evaluate=false').json()]
print(f'alert {aid} in open queue after close?', aid in open_after)

close: {'alert_id': 3, 'status': 'closed'}
alert 3 in open queue after close? False


## 5. The feedback loop improving decisions — suppression
A service account whose flagged logins have consistently been labelled *benign* is a chronic false positive. The same burst that alerted for the attacker is **suppressed** for this subject by the outcome-weighting rule — decisions informed by history, not just the current score.

In [6]:
SVC = 'svc-backup@DOM1'
for i in range(12):
    r = client.post('/identity/score', json={'src_user':SVC,'src_pc':f'C81{i:02d}',
        'auth_type':'TotallyUnrecognizedAuth','logon_type':'RemoteInteractive',
        'orientation':'LogOn','success':'Fail'})
    vid = r.headers.get('X-Verdict-Id')
    if vid: client.post(f'/decision/verdicts/{vid}/feedback', json={'ground_truth':'benign'})
client.post('/decision/evaluate')
svc_alerts = [a for a in client.get('/decision/alerts?limit=100&auto_evaluate=false').json()
              if a['subject']==SVC and a['rule']=='identity_burst']
print(f'identity_burst alerts for the attacker ({ATTACKER}):',
      len([a for a in alerts if a['subject']==ATTACKER and a['rule']=='identity_burst']))
print(f'identity_burst alerts for the benign-history svc acct ({SVC}):', len(svc_alerts),
      '  <- suppressed')

identity_burst alerts for the attacker (ANONYMOUS LOGON@C9999): 1
identity_burst alerts for the benign-history svc acct (svc-backup@DOM1): 0   <- suppressed


## Takeaway
The pipeline turns a raw stream into **triaged, actioned, auditable decisions**: rules raise alerts from the recorded trail, responders act on them and log every response, analysts close them, and the ground-truth feedback loop suppresses chronic false positives so the alert queue stays trustworthy. Reproduces `backend/policy.py`, `backend/actions.py`, and the `/decision` alert/action/close endpoints.